# Volatility Regression 01 — full-feature, next-five-minute OLS (v0.1.2)

**Question.** How much of the **volatility of the next five-minute bar** does a pure OLS on the current bar's causal features explain, and which predictors can be removed without worsening out-of-sample forecasts?

**Targets** (both volatility, one subsequent bar, never across a gap or an 08:00 session change):

- `parkinson`: $\sigma^{P}_{t+1}=\log(H_{t+1}/L_{t+1})/\sqrt{4\log 2}$ from the next perpetual **trade** bar.
- `index_abs`: $\sigma^{I}_{t+1}=|r^{\text{index}}_{t+1}|/\sqrt{2/\pi}$ from the next Deribit **index** 5m log return.

Response: $Y_{t+1}=\log(\sigma_{t+1}+\epsilon_{\text{train}})$. By default the paired experiment requires BOTH current- and next-bar estimators at the origin, and asserts identical complete-case/scored rows and eligible predictor sets across both targets. The estimator comparison is therefore made on matching observations. If a feature (e.g. trade count, spread) carries weight under `parkinson` but not under `index_abs`, its weight is measurement of the trade range, not volatility.

**Models.** Full OLS (catalog features with sufficient joint coverage) and reduced OLS (greedy backward deletion on the selection split). No HAR baseline. Two parameter-free references on the same rows: **train mean** ($R^2_{\text{OOS}}$ benchmark) and **persistence** $\hat Y_{t+1}=\log(\sigma_t+\epsilon)$.

**Splits.** Development expiries only (≤ 2026-10-07; HSMM 06 confirmation untouched), chronological by session ≈ 60/20/20. The test split is scored once.

v0.1.1 changes follow Volatility Regression 002 (Ivan approval, 2026-10-08). v0.1.2 (Volatility Regression 004) corrects and asserts the *final* common observation masks. No model family or target definition changed.

## 1. Environment and data (DuckDB read-only)

In [1]:
from pathlib import Path
import os, sys, json, subprocess
import numpy as np
import pandas as pd
import duckdb
from dotenv import load_dotenv

HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / "src" / "data_engineering").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Launch from the repository root or one of its subfolders.")
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")
DATA_ROOT = os.getenv("DERIBIT_DATA_ROOT")
if not DATA_ROOT:
    raise RuntimeError("DERIBIT_DATA_ROOT must be set in .env.")
DB_PATH = Path(DATA_ROOT) / "database" / "research.duckdb"
if not DB_PATH.is_file():
    raise FileNotFoundError(DB_PATH)

from data_engineering.modeling import vol_ols_5m as mod
OUTPUT = ROOT / "notebooks" / "Volatility_Regression" / "01"   # outputs live in a numbered subfolder
OUTPUT.mkdir(parents=True, exist_ok=True)

DEVELOPMENT_END = "2026-10-07"   # HSMM 06 confirmation starts after this expiry
COVERAGE_MIN = 0.50
MAX_DROPS = 15
MIN_GAIN = 0.0
REQUIRE_BOTH_TARGETS = True

with duckdb.connect(str(DB_PATH), read_only=True) as con:
    con.execute("SET TimeZone = 'UTC'")
    df = con.execute(
        """
        SELECT * FROM time_series_feature_state
        WHERE TRY_CAST(expiry_date AS DATE) <= CAST(? AS DATE)
        ORDER BY underlying, state_time
        """, [DEVELOPMENT_END]).df()
ASSETS = tuple(sorted(df["underlying"].dropna().astype(str).unique()))
print("Vol-OLS version:", mod.VOL_OLS_5M_VERSION)
print("Assets:", ASSETS, "| rows:", len(df))
print("Span:", df.state_time.min(), "to", df.state_time.max())

Vol-OLS version: 0.1.2
Assets: ('AVAX', 'BTC', 'ETH', 'HYPE', 'SOL', 'TRX', 'XRP') | rows: 36288
Span: 2026-09-19 08:00:00+00:00 to 2026-10-07 07:55:00+00:00


In [2]:
# Synthetic alignment / estimator tests; abort on failure.
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(ROOT / "tests"),
                "-p", "test_vol_ols_5m.py", "-v"], cwd=ROOT, check=True)

CompletedProcess(args=['D:\\Anaconda\\envs\\deribit-data\\python.exe', '-m', 'unittest', 'discover', '-s', 'D:\\Projects\\deribit_market_data\\tests', '-p', 'test_vol_ols_5m.py', '-v'], returncode=0)

## 2. Target-quality audit

On development rows where both targets are valid. Things to read:

- `share_zero_range`, `share_trades_le_1`: zero-range bars put $Y$ at $\log\epsilon$, a mass point far below the median.
- `share_range_le_spread`: next-bar range no larger than the spread, i.e. plausibly bounce.
- `corr_log_parkinson_log_index_abs`: agreement between the two estimators of the same volatility.
- `corr_log_parkinson_log1p_trades` vs `corr_log_index_abs_log1p_trades`: how much of the trade-range target follows trade count beyond what index volatility does.

In [3]:
audit_rows = []
for asset in ASSETS:
    g = df[df["underlying"] == asset]
    meta, _, _ = mod.make_panel(g)
    both = (meta["sigma_next_5m"].notna() & meta["index_sigma_next_5m"].notna()).to_numpy()
    audit_rows.append({"underlying": asset, **mod.target_audit(meta, both)})
audit_df = pd.DataFrame(audit_rows)
display(audit_df.round(3))

,underlying,n,share_zero_range,share_index_zero,share_trades_le_1,trade_count_p10,trade_count_p50,trade_count_p90,share_range_le_spread,n_both_positive,corr_log_parkinson_log_index_abs,spearman_parkinson_index_abs,corr_log_parkinson_log1p_trades,corr_log_parkinson_log_spread,corr_log_index_abs_log1p_trades
0,AVAX,4067,0.100,0.015,0.072,2.0,11.0,57.4,0.152,3608,0.487,0.573,0.671,0.226,0.334
1,BTC,4407,0.000,0.000,0.000,122.0,510.0,1532.4,0.001,4406,0.644,0.694,0.750,0.567,0.413
2,ETH,4407,0.000,0.002,0.000,60.0,266.0,731.4,0.000,4400,0.636,0.691,0.703,0.546,0.367
3,HYPE,3640,0.185,0.002,0.137,1.0,7.0,41.0,0.270,2959,0.383,0.507,0.539,0.138,0.157
4,SOL,4407,0.000,0.001,0.000,64.6,202.0,465.0,0.000,4402,0.585,0.657,0.662,0.666,0.292
5,TRX,1404,0.488,0.004,0.290,1.0,3.0,54.7,0.757,716,0.351,0.408,0.263,-0.025,0.010
6,XRP,4119,0.075,0.019,0.050,2.0,16.0,96.0,0.107,3746,0.519,0.590,0.693,0.369,0.310


## 3. Full and reduced OLS per asset and target

In [4]:
stats, coverage_rows, inventory_frames, coef_rows = [], [], [], []
elim_frames, clock_frames, errors = [], [], []

for asset in ASSETS:
    g = df[df["underlying"] == asset]
    # Prepare both regressions BEFORE fitting either. A mismatched origin set
    # is a hard error, not a silent cross-target coefficient comparison.
    try:
        panels = {
            target: mod.prepare_asset(
                g, target, coverage=COVERAGE_MIN,
                development_end=DEVELOPMENT_END,
                require_both_targets=REQUIRE_BOTH_TARGETS
            ) for target in mod.TARGETS
        }
        if REQUIRE_BOTH_TARGETS:
            mod.assert_paired_samples(panels["parkinson"], panels["index_abs"])
    except Exception as exc:
        reason = f"{type(exc).__name__}: {exc}"
        for target in mod.TARGETS:
            errors.append({"underlying": asset, "target": target,
                           "reason": f"Paired preparation failed: {reason}"})
        print(f"{asset:>6} paired preparation NOT FIT: {reason}")
        continue

    for target in mod.TARGETS:
        try:
            d = panels[target]
            X, y, names = d["X"], d["y"], d["features"]
            tr, sel, tst = (d[s] & d["common"] for s in ("train", "selection", "test"))
            if tr.sum() <= len(names) + 20 or sel.sum() < 20 or tst.sum() < 20:
                raise ValueError(f"Insufficient common rows: train={tr.sum()}, "
                                 f"selection={sel.sum()}, test={tst.sum()}, p={len(names)}")
            mu = d["train_mean"]
            full = mod.fit_ols(X, y, tr, names)
            reduced_names, hist = mod.backward_validation(
                X, y, tr, sel, names, max_drops=MAX_DROPS, min_improvement=MIN_GAIN)
            reduced = mod.fit_ols(X, y, tr, reduced_names)

            for split, mask in (("train", tr), ("selection", sel), ("test", tst)):
                idx = np.flatnonzero(mask)
                preds = {
                    "train_mean": np.full(len(idx), mu),
                    "persistence": d["persistence"][idx],
                    "full_ols": full.predict(X.iloc[idx]),
                    "reduced_ols": reduced.predict(X.iloc[idx]),
                }
                for model, pred in preds.items():
                    p = {"full_ols": len(full.names), "reduced_ols": len(reduced.names)}.get(model, 0)
                    cond = {"full_ols": full.condition_number,
                            "reduced_ols": reduced.condition_number}.get(model, np.nan)
                    stats.append({"underlying": asset, "target": target, "model": model,
                                  "features": p, "condition_number": cond,
                                  "epsilon_train": d["epsilon"],
                                  **mod.score(y[idx], pred, mu, split)})

            for fit_name, fit in (("full_ols", full), ("reduced_ols", reduced)):
                for name, beta in zip(fit.names, fit.beta[1:]):
                    coef_rows.append({"underlying": asset, "target": target, "model": fit_name,
                                      "feature": name, "beta_per_train_sd": float(beta),
                                      "condition_number": fit.condition_number})

            inv = d["inventory"].copy()
            inv["underlying"], inv["target"] = asset, target
            inv["train_coverage"] = inv["name"].map(d["coverage"])
            inv["full_ols"] = inv["name"].isin(names)
            inv["reduced_ols"] = inv["name"].isin(reduced_names)
            inv["low_coverage"] = inv["name"].isin(d["excluded_coverage"])
            inv["low_joint_coverage"] = inv["name"].isin(d["excluded_joint_coverage"])
            inv["aliased_or_constant"] = inv["name"].isin(d["excluded_aliases"])
            inventory_frames.append(inv)

            coverage_rows.append({
                "underlying": asset, "target": target,
                "target_valid_train": int(d["train"].sum()),
                "scored_train": int(tr.sum()), "scored_selection": int(sel.sum()),
                "scored_test": int(tst.sum()), "n_sessions": len(d["sessions"]),
                "catalog_columns": d["X"].shape[1], "full_p": len(names),
                "reduced_p": len(reduced_names), "max_drops_reached": len(hist) >= MAX_DROPS,
                "removed_for_coverage": len(d["excluded_coverage"]),
                "removed_for_joint_coverage": len(d["excluded_joint_coverage"]),
                "removed_for_alias": len(d["excluded_aliases"])})
            cc = mod.clock_coverage(d)
            cc["underlying"], cc["target"] = asset, target
            clock_frames.append(cc)
            if len(hist):
                hist["underlying"], hist["target"] = asset, target
                elim_frames.append(hist)
            print(f"{asset:>6} {target:<10} n={int(tr.sum())}/{int(sel.sum())}/{int(tst.sum())} "
                  f"p={len(names)} -> {len(reduced_names)}")
        except Exception as exc:
            errors.append({"underlying": asset, "target": target,
                           "reason": f"{type(exc).__name__}: {exc}"})
            print(f"{asset:>6} {target:<10} NOT FIT: {errors[-1]['reason']}")

metrics_df = pd.DataFrame(stats)
coverage_df = pd.DataFrame(coverage_rows)
inventory_df = pd.concat(inventory_frames, ignore_index=True) if inventory_frames else pd.DataFrame()
coefficients_df = pd.DataFrame(coef_rows)
elimination_df = pd.concat(elim_frames, ignore_index=True) if elim_frames else pd.DataFrame()
clock_df = pd.concat(clock_frames, ignore_index=True) if clock_frames else pd.DataFrame()
errors_df = pd.DataFrame(errors)
display(coverage_df)
if not errors_df.empty:
    display(errors_df)

  AVAX parkinson  n=765/145/207 p=69 -> 54
  AVAX index_abs  n=765/145/207 p=69 -> 54
   BTC parkinson  n=529/90/165 p=63 -> 48
   BTC index_abs  n=529/90/165 p=63 -> 48
   ETH parkinson  n=506/63/111 p=68 -> 53
   ETH index_abs  n=506/63/111 p=68 -> 53
  HYPE parkinson  n=371/84/151 p=69 -> 54
  HYPE index_abs  n=371/84/151 p=69 -> 54
   SOL parkinson  n=441/29/33 p=67 -> 52
   SOL index_abs  n=441/29/33 p=67 -> 52
   TRX parkinson  NOT FIT: ValueError: Insufficient common rows: train=175, selection=33, test=13, p=48
   TRX index_abs  NOT FIT: ValueError: Insufficient common rows: train=175, selection=33, test=13, p=48
   XRP parkinson  n=521/53/20 p=69 -> 54
   XRP index_abs  n=521/53/20 p=69 -> 54


,underlying,target,target_valid_train,scored_train,scored_selection,scored_test,n_sessions,catalog_columns,full_p,reduced_p,max_drops_reached,removed_for_coverage,removed_for_joint_coverage,removed_for_alias
0,AVAX,parkinson,2298,765,145,207,16,82,69,54,True,9,0,4
1,AVAX,index_abs,2298,765,145,207,16,82,69,54,True,9,0,4
2,BTC,parkinson,2461,529,90,165,16,82,63,48,True,15,0,4
3,BTC,index_abs,2461,529,90,165,16,82,63,48,True,15,0,4
4,ETH,parkinson,2461,506,63,111,16,82,68,53,True,9,0,5
5,ETH,index_abs,2461,506,63,111,16,82,68,53,True,9,0,5
6,HYPE,parkinson,1520,371,84,151,16,82,69,54,True,9,0,4
7,HYPE,index_abs,1520,371,84,151,16,82,69,54,True,9,0,4
8,SOL,parkinson,2459,441,29,33,16,82,67,52,True,11,0,4
9,SOL,index_abs,2459,441,29,33,16,82,67,52,True,11,0,4


,underlying,target,reason
0,TRX,parkinson,ValueError: Insufficient common rows: train=17...
1,TRX,index_abs,ValueError: Insufficient common rows: train=17...


## 4. Test scores (scored once)

RMSE ratio is relative to persistence on the same rows: below 1 means the model beats "next bar = current bar".

In [5]:
if not metrics_df.empty:
    test = metrics_df[metrics_df["split"] == "test"].copy()
    ref = test[test["model"] == "persistence"].set_index(["underlying", "target"])["log_rmse"]
    test["rmse_vs_persistence"] = test["log_rmse"] / test.set_index(
        ["underlying", "target"]).index.map(ref).to_numpy()
    display(test.pivot_table(index=["underlying", "target"], columns="model",
                             values=["r2_vs_train_mean", "rmse_vs_persistence"]).round(3))

r2_vs_train_mean                                     \
model                        full_ols persistence reduced_ols train_mean   
underlying target                                                          
AVAX       index_abs           -0.304      -0.256       0.141        0.0   
           parkinson            0.129       0.216       0.307        0.0   
BTC        index_abs           -1.075      -0.689      -1.492        0.0   
           parkinson            0.145       0.281       0.042        0.0   
ETH        index_abs           -8.060      -0.429     -11.535        0.0   
           parkinson           -9.329       0.631      -3.482        0.0   
HYPE       index_abs           -0.046      -0.622       0.140        0.0   
           parkinson           -0.151      -0.409      -0.215        0.0   
SOL        index_abs            0.030      -0.373       0.134        0.0   
           parkinson            0.690       0.733       0.702        0.0   
XRP        index_abs           -0.061      -0.497       0.227        0.0   
           parkinson            0.194       0.143       0.160        0.0   

                     rmse_vs_persistence                                     
model                           full_ols persistence reduced_ols train_mean  
underlying target                                                            
AVAX       index_abs               1.019         1.0       0.827      0.892  
           parkinson               1.054         1.0       0.940      1.130  
BTC        index_abs               1.108         1.0       1.215      0.769  
           parkinson               1.091         1.0       1.155      1.180  
ETH        index_abs               2.518         1.0       2.962      0.836  
           parkinson               5.293         1.0       3.486      1.647  
HYPE       index_abs               0.803         1.0       0.728      0.785  
           parkinson               0.904         1.0       0.929      0.843  
SOL        index_abs               0.840         1.0       0.794      0.853  
           parkinson               1.078         1.0       1.057      1.935  
XRP        index_abs               0.842         1.0       0.719      0.817  
           parkinson               0.970         1.0       0.990      1.080

## 5. Coefficients, elimination history, clock coverage

Coefficients are per one training standard deviation of each feature. With many correlated regressors they are **descriptive only**; check the condition number. Compare each feature across the two targets.

In [6]:
if not coefficients_df.empty:
    red = coefficients_df[coefficients_df["model"] == "reduced_ols"]
    display(red.pivot_table(index=["underlying", "feature"], columns="target",
                            values="beta_per_train_sd").round(3))
if not elimination_df.empty:
    display(elimination_df)
if not clock_df.empty:
    print("Scored share of target-valid rows by UTC hour (train split):")
    display(clock_df[clock_df["split"] == "train"].pivot_table(
        index="utc_hour", columns=["underlying", "target"], values="scored_share").round(2))

target                  index_abs  parkinson
underlying feature                          
AVAX       atm_iv_d15       0.041      0.022
           atm_iv_d30         NaN     -0.040
           atm_iv_d5        0.003      0.050
           atm_iv_d60         NaN      0.035
           bf25_d5          0.024      0.058
...                           ...        ...
XRP        skew025_d5      -0.039      0.016
           skew025_d60        NaN     -0.094
           skew_050         0.036     -0.312
           tfi             -0.018     -0.041
           weekend         -0.403     -0.406

[384 rows x 2 columns]

,step,removed,p_before,rmse_before,rmse_after,improvement,underlying,target
0,1,log1p_tte_min,69,2.205315,2.115421,0.089895,AVAX,parkinson
1,2,weekend,68,2.115421,2.072306,0.043115,AVAX,parkinson
2,3,log_option_spread_med,67,2.072306,2.045202,0.027104,AVAX,parkinson
3,4,perp_mark_basis_bps,66,2.045202,2.018129,0.027073,AVAX,parkinson
4,5,perp_mid_basis_bps,65,2.018129,1.969240,0.048889,AVAX,parkinson
...,...,...,...,...,...,...,...,...
175,11,log_fit_atm_iv,59,0.987744,0.981715,0.006029,XRP,index_abs
176,12,skew025_d60,58,0.981715,0.975671,0.006044,XRP,index_abs
177,13,ret_1h,57,0.975671,0.969502,0.006170,XRP,index_abs
178,14,close_vs_vwap_bps,56,0.969502,0.962560,0.006942,XRP,index_abs


Scored share of target-valid rows by UTC hour (train split):


underlying      AVAX                 BTC                 ETH            \
target     index_abs parkinson index_abs parkinson index_abs parkinson   
utc_hour                                                                 
0               0.40      0.40      0.15      0.15      0.11      0.11   
1               0.35      0.35      0.11      0.11      0.01      0.01   
2               0.19      0.19      0.09      0.09      0.00      0.00   
3               0.06      0.06      0.11      0.11      0.00      0.00   
4               0.03      0.03      0.05      0.05      0.00      0.00   
5               0.01      0.01      0.00      0.00      0.00      0.00   
6               0.00      0.00      0.00      0.00      0.00      0.00   
7               0.00      0.00      0.00      0.00      0.00      0.00   
8               0.00      0.00      0.00      0.00      0.00      0.00   
9               0.30      0.30      0.35      0.35      0.32      0.32   
10              0.31      0.31      0.27      0.27      0.34      0.34   
11              0.30      0.30      0.22      0.22      0.33      0.33   
12              0.30      0.30      0.23      0.23      0.34      0.34   
13              0.44      0.44      0.32      0.32      0.44      0.44   
14              0.56      0.56      0.44      0.44      0.55      0.55   
15              0.61      0.61      0.47      0.47      0.67      0.67   
16              0.71      0.71      0.48      0.48      0.61      0.61   
17              0.43      0.43      0.24      0.24      0.19      0.19   
18              0.36      0.36      0.21      0.21      0.11      0.11   
19              0.53      0.53      0.25      0.25      0.23      0.23   
20              0.45      0.45      0.27      0.27      0.22      0.22   
21              0.44      0.44      0.22      0.22      0.05      0.05   
22              0.55      0.55      0.40      0.40      0.17      0.17   
23              0.61      0.61      0.26      0.26      0.24      0.24   

underlying      HYPE                 SOL                 XRP            
target     index_abs parkinson index_abs parkinson index_abs parkinson  
utc_hour                                                                
0               0.21      0.21      0.15      0.15      0.27      0.27  
1               0.02      0.02      0.06      0.06      0.15      0.15  
2               0.02      0.02      0.01      0.01      0.13      0.13  
3               0.00      0.00      0.04      0.04      0.15      0.15  
4               0.00      0.00      0.00      0.00      0.09      0.09  
5               0.00      0.00      0.00      0.00      0.01      0.01  
6               0.00      0.00      0.00      0.00      0.00      0.00  
7               0.00      0.00      0.00      0.00      0.00      0.00  
8               0.00      0.00      0.00      0.00      0.00      0.00  
9               0.23      0.23      0.18      0.18      0.32      0.32  
10              0.14      0.14      0.22      0.22      0.18      0.18  
11              0.21      0.21      0.27      0.27      0.19      0.19  
12              0.25      0.25      0.23      0.23      0.16      0.16  
13              0.31      0.31      0.26      0.26      0.33      0.33  
14              0.48      0.48      0.33      0.33      0.47      0.47  
15              0.48      0.48      0.37      0.37      0.60      0.60  
16              0.64      0.64      0.46      0.46      0.68      0.68  
17              0.36      0.36      0.17      0.17      0.38      0.38  
18              0.38      0.38      0.27      0.27      0.20      0.20  
19              0.43      0.43      0.33      0.33      0.27      0.27  
20              0.30      0.30      0.19      0.19      0.20      0.20  
21              0.33      0.33      0.11      0.11      0.26      0.26  
22              0.58      0.58      0.29      0.29      0.43      0.43  
23              0.34      0.34      0.34      0.34      0.33      0.33

## 6. Save outputs (no database writes)

In [7]:
outputs = {
    "01_target_audit.csv": audit_df,
    "01_scores.csv": metrics_df,
    "01_coverage.csv": coverage_df,
    "01_clock_coverage.csv": clock_df,
    "01_feature_inventory.csv": inventory_df,
    "01_coefficients.csv": coefficients_df,
    "01_elimination_history.csv": elimination_df,
    "01_fit_errors.csv": errors_df,
}
for name, table in outputs.items():
    table.to_csv(OUTPUT / name, index=False)
    print(name, "rows:", len(table))
summary = {
    "version": mod.VOL_OLS_5M_VERSION,
    "targets": {"parkinson": "next 5m perpetual trade-range Parkinson volatility",
                "index_abs": "|next 5m index log return| / sqrt(2/pi)"},
    "transformed_target": "log(sigma_next_5m + train_only_epsilon)",
    "require_both_targets": REQUIRE_BOTH_TARGETS,
    "references": ["train_mean", "persistence"],
    "r2_benchmark": "training mean",
    "development_last_expiry": DEVELOPMENT_END,
    "coverage_threshold": COVERAGE_MIN,
    "max_backward_drops": MAX_DROPS,
    "fitted": [f"{r['underlying']}/{r['target']}" for r in coverage_rows],
    "failed": errors,
    "claim": "development experiment only; no held-out confirmation",
}
(OUTPUT / "01_run_metadata.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
# Remove v0.1.0 output superseded by 01_coefficients.csv, if present.
old = OUTPUT / "01_full_ols_coefficients.csv"
if old.exists():
    old.unlink()
print("Done. Commit the notebook and 01_* outputs.")

01_target_audit.csv rows: 7
01_scores.csv rows: 144
01_coverage.csv rows: 12
01_clock_coverage.csv rows: 864
01_feature_inventory.csv rows: 984
01_coefficients.csv rows: 1440
01_elimination_history.csv rows: 180
01_fit_errors.csv rows: 2
Done. Commit the notebook and 01_* outputs.


## Interpretation contract

1. Exclusion for coverage or collinearity is **not** evidence of zero predictive content.
2. Backward deletion uses the selection split only (≈3 sessions); removals with tiny gains are likely noise. The test split is a one-time diagnostic, not a tuning input.
3. A feature whose coefficient appears under `parkinson` but not under `index_abs` is measuring the trade range (trade count, bounce), not volatility.
4. Rows are complete-case; check `01_clock_coverage.csv` for hours (08:00–09:00 UTC, settlement window) that the scored sample under-represents.
5. Research diagnostics only; HSMM 06 frozen spec and confirmation window untouched.